# Exercise 3: Transliteration by hand

*Scripts and Encoding · version 0.1.0*

> Using the toolkit's Kazakh mapping (based on the 2021 proposal; print the table from `Transliterator("kaz", Script.CYRILLIC, Script.LATIN)`), convert *Алматы*, *Қазақстан*, *Астана* and *киім* ('clothes') by hand, then check your answers with the toolkit. Identify one mapping whose reverse is ambiguous and explain what additional information would be needed to convert back. Why does implementing a mapping say nothing about its legal status?

**Setup.** `pip install "turkicnlp==0.5.0"`. Cells that fetch data from the web need internet access.

In [ ]:
from collections import defaultdict
from turkicnlp.scripts import Script
from turkicnlp.scripts.transliterator import Transliterator

to_latin = Transliterator("kaz", Script.CYRILLIC, Script.LATIN)
to_cyrillic = Transliterator("kaz", Script.LATIN, Script.CYRILLIC)

table = to_latin._forward_map          # private attribute in 0.5.0: the character table
print(" ".join(f"{k}→{v or '∅'}" for k, v in sorted(table.items()) if k.islower()))

In [ ]:
# Reverse ambiguity: Latin outputs produced by more than one Cyrillic letter
sources = defaultdict(list)
for cyr, lat in table.items():
    if cyr.islower():
        sources[lat].append(cyr)
print({lat or "∅ (deleted)": cyrs for lat, cyrs in sources.items() if len(cyrs) > 1})

for word in ["Алматы", "Қазақстан", "Астана", "киім"]:
    latin = to_latin.transliterate(word)
    print(f"{word:<10} → {latin:<10} → back: {to_cyrillic.transliterate(latin)}")

## Worked solution (toolkit mapping, TurkicNLP 0.5.0)

| Cyrillic | By hand with the table | Toolkit |
|---|---|---|
| *Алматы* | А→A, л→l, м→m, а→a, т→t, ы→y | *Almaty* |
| *Қазақстан* | Қ→Q, а→a, з→z, а→a, қ→q, с→s, т→t, а→a, н→n | *Qazaqstan* |
| *Астана* | | *Astana* |
| *киім* | к→k, и→i, і→ı, м→m | *kiım* |

In this mapping Kazakh *і* becomes dotless *ı* and *и* becomes dotted *i*, so *киім* is *kiım*.

**Ambiguous reverse mappings.** The table maps several Cyrillic letters to the same Latin output: *и* and *й* both → *i*, *һ* and *х* both → *h*, *е* and *э* both → *e*, and the hard and soft signs *ъ*, *ь* are deleted. Converting *i* back to Cyrillic therefore needs context: the toolkit uses position rules (here *киім* comes back correctly), but in general a lexicon, or the original text, is needed to decide between *и* and *й*, to restore *х* versus *һ* in loanwords, and to put back a deleted soft sign in a Russian loanword.

**Legal status.** A mapping table is a technical decision by the toolkit's authors. Whether the 2021 proposal, a later revision or a commission recommendation is the legally adopted alphabet is a question for the official acts (see the chapter's section on active transitions); implementing one does not make it official, and record the mapping version with any converted corpus.